# 로컬 실행용 정리 사본
2025년 KT AIVLE 미니프로젝트 1차 팀 산출물입니다. 2026-10-28에 실행 경로와 환경 설정만 정리하고 저장 출력을 제거했습니다. 알고리즘·실험 오류는 당시 상태를 보존했습니다. 전체 재실행을 검증한 버전은 아닙니다. 세부 한계는 ../../docs/KNOWN_ISSUES.md를 확인하세요.

#**스마트폰 센서 데이터 기반 모션 분류**
# 단계3 : 단계별 모델링


## 0.미션4

* 단계별로 나눠서 모델링을 수행하고자 합니다.  
* 단계 구분 예시
    * 단계1 : 정적(0), 동적(1) 행동 분류 모델 생성
    * 단계2 : 세부 동작에 대한 분류모델 생성
        * 단계1 모델에서 0으로 예측 -> 정적 행동 3가지 분류 모델링
        * 단계1 모델에서 1으로 예측 -> 동적 행동 3가지 분류 모델링
* (선택) 모델 통합
    * 두 단계 모델을 통합하고, 새로운 데이터에 대해서 최종 예측결과와 성능평가가 나오도록 함수로 만들기
* 성능 비교
    * 기본 모델링의 성능과 비교
    * 성능 가이드
        * Accuracy : 0.97 ~ 0.99
* (선택) 파이프라인 구성
    * test 데이터를 입력하여, 전처리 및 예측결과가 나오도록 함수 구성

## 1.환경설정

* 세부 요구사항
    - 경로 설정 : 구글콜랩
        * 구글 드라이브 바로 밑에 project1 폴더를 만들고,
        * 데이터 파일을 복사해 넣습니다.
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
        * 필요하다고 판단되는 라이브러리를 추가하세요.


### (1) 경로 설정

* 구글 드라이브 연결

In [ ]:
# 로컬 실행용 사본: Google Drive 마운트 생략


In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'README.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
path = str(ROOT / 'data' / 'har') + '/'


In [ ]:
import os
os.chdir(path)


### (2) 라이브러리 불러오기

* 라이브러리 로딩
  - pandas, numpy,matplotlib,seaborn, joblib, 모델링에 필요한 라이브러리를 로딩합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import *

from keras.models import Sequential
from keras.layers import Dense, Flatten, Dropout, Input
from keras.backend import clear_session
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

In [ ]:
# 학습곡선 함수
def dl_history_plot(history):
    plt.figure(figsize=(10,6))
    plt.plot(history['loss'], label='train_err', marker = '.')
    plt.plot(history['val_loss'], label='val_err', marker = '.')

    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid()
    plt.show()

### (3) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
    * data01_test.csv : 테스트용
    * feature.csv : feature 이름을 계층구조로 정리한 데이터

* 세부 요구사항
    * 칼럼 삭제 : data01_train.csv와 data01_test.csv 에서 'subject' 칼럼은 불필요하므로 삭제합니다.

#### 1) 데이터로딩

In [ ]:
# 주어진 데이터 셋을 불러오세요.(3개)
train =  pd.read_csv('data01_train.csv')
test =  pd.read_csv('data01_test.csv')
features = pd.read_csv('features.csv')

In [ ]:
#불필요한 칼럼을 삭제하세요.

In [ ]:
train = train.drop('subject', axis=1)
test = test.drop('subject', axis=1)

#### 2) 기본 정보 조회

In [ ]:
#전체 데이터의 행,열 개수 확인
for i in [train, test, features]:
    print(i.shape)

In [ ]:
#전체 데이터의 상위 5개 행 확인
for i in [train, test, features]:
    display(i.head(5))

In [ ]:
#전체 데이터의 수치형 변수 분포 확인
for i in [train, test, features]:
    display(i.describe())

In [ ]:
#전체 데이터의 모든 변수 확인
for i in [train, test, features]:
    display(i.info())

## 2.데이터 전처리

* 세부 요구사항
    - Label 추가 : 1단계 모델을 위한 레이블 추가
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

### (1) 1단계 모델링을 위한 레이블

In [ ]:
#Lable 추가(1단계 모델:정적(0), 동적(1) 행동 분류 모델 생성 )
train['is_dynamic'] = train.loc[:, 'Activity'].map(lambda a: 0 if a in ['LAYING', 'STANDING', 'SITTING'] else 1)

In [ ]:
train['is_dynamic'].value_counts()

### (2) x, y 분리

In [ ]:
#x,y 분리하기
X = train.drop(['Activity', 'is_dynamic'], axis=1)
y = train.loc[:, 'is_dynamic']

### (3) 데이터 분할
* train, val 분할

In [ ]:
#데이터 분할 진행(train:val = 8:2 혹은 7:3 권장)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

### (4) 스케일링


* 세부 요구사항
    - 스케일링을 필요로 하는 알고리즘 사용을 위해서 코드 수행
    - min-max 방식 혹은 standard 방식 중 한가지 사용.

In [ ]:
#스케일링 방식을 선택해서 스케일링을 진행합니다.
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

## **3.단계별 모델링**

### (1) 단계1

* 세부 요구사항
    * 적절한 단계로 구분한 후, 1단계를 분류하는 모델 생성
        * 예시 : 정적 행동(Laying, Sitting, Standing)과 동적 행동(동적 : Walking, Walking-Up, Walking-Down)을 구분하는 모델 생성.
    * 몇 가지 모델을 만들고 가장 성능이 좋은 모델을 선정하시오.(기본 모델링 참고)

#### 1) 모델1

In [ ]:
n_features = X.shape[1]
es = EarlyStopping(monitor='val_loss', min_delta=0.005, patience=10, verbose=1)

In [ ]:
# 모델 설계
clear_session()
model1 = Sequential([
    Input(shape = (n_features, )),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(6, activation='softmax')
])
model1.summary()

In [ ]:
# 컴파일 및 학습
model1.compile(optimizer=Adam(learning_rate=0.01), loss = 'sparse_categorical_crossentropy')
hist = model1.fit(X_train, y_train, epochs=100, validation_split=0.2, verbose=0, callbacks=[es]).history

In [ ]:
# 학습곡선
dl_history_plot(hist)

In [ ]:
# 예측 및 평가
y_pred = model1.predict(X_val)
y_pred =np.argmax(y_pred, axis=1)
print(classification_report(y_val, y_pred))

#### 2) 모델2

In [ ]:
# 모델 설계

In [ ]:
# 컴파일 및 학습

In [ ]:
# 학습곡선

In [ ]:
# 예측 및 평가

### (2) 단계2

#### 1) 단계2-1 : 정적 동작 세부 분류

* 세부 요구사항
    * 정적 행동(Laying, Sitting, Standing)인 데이터 추출
    * Laying, Sitting, Standing 를 분류하는 모델을 생성
    * 몇가지 모델을 만들고 가장 성능이 좋은 모델을 선정하시오.

In [ ]:
# (tip) 정적 행동(0)인 데이터 추출
stop_X = X.loc[y.values == 0, :]
stop_y = train.loc[y.values == 0, 'Activity']

In [ ]:
# (tip) 인코딩 진행, map 활용해서 숫자레이블로 매핑
le = LabelEncoder()
stop_y = le.fit_transform(stop_y)
X_train, X_val, y_train, y_val = train_test_split(stop_X, stop_y, test_size=0.2, random_state=42)

In [ ]:
# 모델 설계
clear_session()
stop_model = Sequential([
    Input(shape = (n_features, )),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(8, activation='relu'),
    Dense(3, activation='softmax')
])
stop_model.summary()

In [ ]:
# 컴파일 및 학습
stop_model.compile(optimizer=Adam(learning_rate=0.01), loss = 'sparse_categorical_crossentropy')
hist = stop_model.fit(X_train, y_train, epochs=100, validation_split=0.2, verbose=0, callbacks=[es]).history

In [ ]:
# 학습곡선
dl_history_plot(hist)

In [ ]:
# 예측 및 평가
y_pred = stop_model.predict(X_val)
y_pred =np.argmax(y_pred, axis=1)
print(classification_report(y_val, y_pred))

#### 2) 단계2-2 : 동적 동작 세부 분류

* 세부 요구사항
    * 동적 행동(Walking, Walking Upstairs, Walking Downstairs)인 데이터 추출
    * Walking, Walking Upstairs, Walking Downstairs 를 분류하는 모델을 생성
    * 몇가지 모델을 만들고 가장 성능이 좋은 모델을 선정하시오.

In [ ]:
# (tip) 정적 행동(0)인 데이터 추출
dynamic_X = X.loc[y.values == 1, :]
dynamic_y = train.loc[y.values == 1, 'Activity']

In [ ]:
# (tip) 인코딩 진행, map 활용해서 숫자레이블로 매핑
le = LabelEncoder()
dynamic_y = le.fit_transform(dynamic_y)
X_train, X_val, y_train, y_val = train_test_split(dynamic_X, dynamic_y, test_size=0.2, random_state=42)

In [ ]:
# 모델 설계
clear_session()
dynamic_model = Sequential([
    Input(shape = (n_features, )),
    Dense(64, activation='relu'),
    Dense(64, activation='relu'),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(8, activation='relu'),
    Dense(3, activation='softmax')
])
dynamic_model.summary()

In [ ]:
# 컴파일 및 학습
dynamic_model.compile(optimizer=Adam(learning_rate=0.001), loss = 'sparse_categorical_crossentropy')
hist = dynamic_model.fit(X_train, y_train, epochs=100, validation_split=0.2, verbose=0, callbacks=[es]).history

In [ ]:
# 학습곡선
dl_history_plot(hist)

In [ ]:
# 예측 및 평가
y_pred = dynamic_model.predict(X_val)
y_pred =np.argmax(y_pred, axis=1)
print(classification_report(y_val, y_pred))

### (3) (옵션) 분류 모델 파이프라인 구성


* 세부 요구사항
    * 두 단계 모델을 통합하고, 새로운 데이터(test)에 대해서 최종 예측결과와 성능평가가 나오도록 함수로 만들기
    * 데이터 파이프라인 구축 : test데이터가 로딩되어 전처리 과정을 거치고, 예측 및 성능 평가 수행

* 예시
![](https://github.com/DA4BAM/image/blob/main/pipeline%20function.png?raw=true)

#### 1) 함수 만들기

In [ ]:
def smartphone_pipeline(train, test, stage1, stop_model, dynamic_model):
    es = EarlyStopping(monitor='val_loss', min_delta=0.01, patience=10, verbose=1)
    # 1. 전처리
    # 1-1. 스케일링
    # 1-2. 입력값 만들기
    #-------------------
    train['is_dynamic'] = train.loc[:, 'Activity'].map(lambda x: 0 if x in ['LAYING', 'STANDING', 'SITTING'] else 1)
    test['is_dynamic'] = test.loc[:, 'Activity'].map(lambda x: 0 if x in ['LAYING', 'STANDING', 'SITTING'] else 1)
    X_train = train.drop(['is_dynamic', 'Activity'], axis=1)
    X_test = test.drop(['is_dynamic', 'Activity'], axis=1)
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # 2. 예측하기
    # 2-1. 단계1을 모델로 0,1 구분
    # 2-2. 단계 1의 결과로 데이터 나누기
    # 2-3. 단계2 모델로 예측
    # 2-4. 예측 결과 원래 값으로 변환
    # 2-5. 하나로 합쳐보기
    #-------------------
    # 모델학습1
    y_train = train.loc[:, 'is_dynamic']
    y_test = test.loc[:, 'Activity']

    clear_session()
    stage1.compile(optimizer=Adam(learning_rate=0.01), loss = 'sparse_categorical_crossentropy')
    stage1.fit(X_train, y_train, epochs=100, validation_split=0.2, verbose=0, callbacks=[es])

    stop_inx = (train['is_dynamic'] == 0).values
    X_train_stop = X_train[stop_inx]
    y_train_stop = train.loc[stop_inx, 'Activity']

    dynamic_inx = (train['is_dynamic'] == 1).values
    X_train_dynamic = X_train[dynamic_inx]
    y_train_dynamic = train.loc[dynamic_inx, 'Activity']

    stop_le = LabelEncoder()
    dyn_le  = LabelEncoder()
    y_train_stop = stop_le.fit_transform(y_train_stop)
    y_train_dynamic = dyn_le.fit_transform(y_train_dynamic)
    y_train_stop = pd.Series(y_train_stop)
    y_train_dynamic = pd.Series(y_train_dynamic)

    # 모델 학습2
    stop_model.compile(optimizer=Adam(learning_rate=0.01), loss = 'sparse_categorical_crossentropy')
    stop_model.fit(X_train_stop, y_train_stop, epochs=100, validation_split=0.2, verbose=0, callbacks=[es])
    dynamic_model.compile(optimizer=Adam(learning_rate=0.01), loss = 'sparse_categorical_crossentropy')
    dynamic_model.fit(X_train_dynamic, y_train_dynamic, epochs=100, validation_split=0.2, verbose=0, callbacks=[es])

    # 모델 예측1
    y_pred = stage1.predict(X_test)
    y_pred = y_pred.argmax(axis=1)
    y_pred_full = np.empty(test.shape[0], dtype=object)

    # 모델예측 2-1
    idx_stop = np.where(y_pred == 0)[0]
    y_pred_stop = stop_model.predict(X_test[idx_stop])
    y_pred_stop = y_pred_stop.argmax(axis=1)
    y_pred_full[idx_stop] = stop_le.inverse_transform(y_pred_stop)

    # 모델예측 2-2
    idx_dynamic = np.where(y_pred == 1)[0]
    y_pred_dynamic = dynamic_model.predict(X_test[idx_dynamic])
    y_pred_dynamic = y_pred_dynamic.argmax(axis=1)
    y_pred_full[idx_dynamic] = dyn_le.inverse_transform(y_pred_dynamic)

    y_pred = y_pred_full.astype(str)


    # 3. 최종 성능평가  하기
    return confusion_matrix(y_test, y_pred), classification_report(y_test, y_pred)


In [ ]:
# (옵션) 위 참고해서 분류모델 파이프라인 만들어보기

#### 2) test 셋으로 예측하고 평가하기

In [ ]:
# test 셋의 구조 한번 확인해보고 성능평가 해보기

In [ ]:
print(test.shape)
print(train.shape)

In [ ]:
cm, cr = smartphone_pipeline(train=train, test=test, stage1=model1, stop_model=stop_model, dynamic_model=dynamic_model)
print(cm)
print(cr)